In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import GridSearchCV

Import Dataset

In [2]:
df = pd.read_csv('processed_dataset.csv')
df

,weight,ap_hi,ap_lo,cholesterol,gluc,active,age_years,BMI,BMI_category,pulse_pressure,cardio
0,83.0,120,80,1,1,0,52.041096,31.626276,4,40,1
1,64.0,120,80,1,1,1,47.449315,25.636917,3,40,0
2,95.0,160,100,2,1,1,52.101370,34.894399,4,60,1
3,83.0,150,100,1,1,1,55.857534,30.859607,4,50,1
4,52.0,100,67,1,1,0,49.961644,21.367521,2,33,0
...,...,...,...,...,...,...,...,...,...,...,...
52610,53.0,120,75,1,1,1,57.767123,19.467401,2,45,1
52611,68.0,120,80,1,3,1,60.216438,23.808690,2,40,0
52612,74.0,120,80,1,1,0,54.602740,25.605536,3,40,1
52613,91.0,130,90,1,1,0,57.293151,31.861629,4,40,1


In [3]:
# split the dataset into train and test sets
from sklearn.model_selection import train_test_split
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
normalize_columns = ['weight', 'ap_hi', 'ap_lo', 'age_years', 'BMI', 'pulse_pressure']

# Convert X_train and X_test to DataFrames for easier column-based operations
X_train_df = pd.DataFrame(X_train, columns=df.columns[:-1])
X_test_df = pd.DataFrame(X_test, columns=df.columns[:-1])

# Apply scaling
X_train_df[normalize_columns] = scaler.fit_transform(X_train_df[normalize_columns]) # Fit on training set
X_test_df[normalize_columns] = scaler.transform(X_test_df[normalize_columns]) # Transform test set using SAME SCALER

In [5]:
X_train = X_train_df
X_test = X_test_df
y_train = pd.DataFrame(y_train, columns=['cardio'])
y_test = pd.DataFrame(y_test, columns=['cardio'])

## Decision Tree

In [7]:
from sklearn.tree import DecisionTreeClassifier
dt_params = {
    'max_depth': [10],
    'min_samples_split': [5],
    'min_samples_leaf': [1],
    'criterion': ['gini', 'entropy'],
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                       dt_params, cv=5, scoring='recall')
dt_grid.fit(X_train, y_train)
print("Best Decision Tree Recall:", dt_grid.best_score_)
print("Best Parameters:", dt_grid.best_params_)

Best Decision Tree Recall: 0.6487730234269659
Best Parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5}


## KNN

In [6]:
from sklearn.neighbors import KNeighborsClassifier

knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(KNeighborsClassifier(),
                        knn_params, cv=5, scoring='recall')
knn_grid.fit(X_train, y_train.values.ravel())
print("Best KNN Recall:", knn_grid.best_score_)
print("Best Parameters:", knn_grid.best_params_)

Best KNN Recall: 0.6737037279582838
Best Parameters: {'metric': 'euclidean', 'n_neighbors': 11, 'weights': 'distance'}


## Perceptron

In [ ]:
from sklearn.linear_model import Perceptron

perc_params = {
    'alpha': [1e-4, 1e-3, 1e-2],
    'max_iter': [500, 1000, 5000]
}

perc_grid = GridSearchCV(Perceptron(random_state=42),
                         perc_params, cv=5, scoring='recall')
perc_grid.fit(X_train, y_train.values.ravel())

print("Best Perceptron Recall:", perc_grid.best_score_)
print("Best Parameters:", perc_grid.best_params_)


Best Perceptron Recall: 0.7609728280203409
Best Parameters: {'alpha': 0.001, 'max_iter': 500, 'penalty': 'l2'}


## MLP

In [14]:
from sklearn.neural_network import MLPClassifier

mlp_params = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (50, 50)],
    'activation': ['relu', 'sigmoid'],
    'learning_rate': ['adaptive'],
    'alpha': [1e-5, 1e-4, 1e-3, 1],
    'solver':['adam'],
    'max_iter': [500],
    'early_stopping': [True],
}

mlp_grid = GridSearchCV(MLPClassifier(random_state=42),
                        mlp_params, cv=5, scoring='recall')
mlp_grid.fit(X_train, y_train.values.ravel())
print("Best MLP Recall:", mlp_grid.best_score_)
print("Best Parameters:", mlp_grid.best_params_)


C:\Users\riwki\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
80 fits failed out of a total of 160.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
80 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\riwki\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\riwki\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-pac

Best MLP Recall: 0.698633233087288
Best Parameters: {'activation': 'relu', 'alpha': 1, 'early_stopping': True, 'hidden_layer_sizes': (100, 50), 'learning_rate': 'adaptive', 'max_iter': 500, 'solver': 'adam'}


## SVM

In [ ]:
from sklearn.svm import SVC

param_grid = {
    'C': [0.1, 1, 5, 10],                # ควบคุม margin: ยิ่งมาก = overfit ง่าย
    'kernel': ['linear', 'rbf'],      # 'rbf' ใช้สำหรับ pattern ซับซ้อน
    'gamma': ['scale', 0.01, 0.001],  # ใช้เฉพาะกับ rbf
    'class_weight': [None, 'balanced'],  # จัดการ class imbalance
}

svc = SVC(probability=True, random_state=42)

grid = GridSearchCV(
    svc,
    param_grid,
    scoring='recall',   # หรือ 'f1' ถ้าต้องการ balance
    cv=5,
    verbose=1
)

grid.fit(X_train, y_train.values.ravel())
print("Best Params:", grid.best_params_)
print("Best Recall:", grid.best_score_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


## Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2'],
    'random_state': [42],
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
                       rf_params, cv=5, scoring='recall')
rf_grid.fit(X_train, y_train.values.ravel())
print("Best Random Forest Recall:", rf_grid.best_score_)
print("Best Parameters:", rf_grid.best_params_)

Best Random Forest Recall: 0.677034060387028
Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 200, 'random_state': 42}
